# Baby Step 1 — Execute the First Strategy-Development Operating Loop

**Author:** Alejandro Reynoso  
**Persistent vault:** `/content/drive/MyDrive/Alejandro-Reynoso-Corporate-Civil-Litigation-ExoBrain`

This notebook performs the first complete civil-litigation exo-brain loop for the five active matters created in Baby Step 0:

**observe → normalize → connect → retrieve → analyze → challenge → recommend → decide → record → remember**

It creates Recommendation Version 1 for each active matter and writes all outputs back into the same Google Drive vault.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, csv, datetime, statistics, textwrap
from collections import defaultdict

VAULT = Path(r"/content/drive/MyDrive/Alejandro-Reynoso-Corporate-Civil-Litigation-ExoBrain")
if not VAULT.exists():
    raise FileNotFoundError("Run Baby Step 0 first.")

state_path = VAULT / "00_System" / "Workflow_State.json"
state = json.loads(state_path.read_text(encoding="utf-8"))
if 0 not in state.get("completed_steps", []):
    raise RuntimeError("Baby Step 0 is not complete.")

precedents = json.loads((VAULT/"data"/"precedents.json").read_text(encoding="utf-8"))
matters = json.loads((VAULT/"data"/"active_matters.json").read_text(encoding="utf-8"))

print("Precedents:", len(precedents))
print("Active matters:", len(matters))


## Retrieval model

Every precedent is scored against every active matter using six visible dimensions:

| Dimension | Weight |
|---|---:|
| Doctrinal relevance | 25% |
| Issue relevance | 20% |
| Jurisdictional authority | 20% |
| Court authority | 15% |
| Treatment quality | 10% |
| Recency | 10% |

The score is a retrieval aid, not a legal conclusion.


In [ ]:
WEIGHTS = {
    "doctrinal_relevance": 0.25,
    "issue_relevance": 0.20,
    "jurisdictional_authority": 0.20,
    "court_authority": 0.15,
    "treatment_quality": 0.10,
    "recency": 0.10
}
TREATMENT = {
    "Positive": 100, "Followed": 95, "Distinguished": 60,
    "Limited": 45, "Criticized": 25, "Overruled": 0
}
PRECEDENTIAL = {
    "Binding": 100, "Persuasive": 75, "Limited Persuasive": 55,
    "Unpublished": 35, "Arbitral Persuasive": 45
}
assert abs(sum(WEIGHTS.values())-1.0) < 1e-9

(VAULT/"00_System"/"Baby_Step_1_Retrieval_Model.json").write_text(
    json.dumps({
        "weights": WEIGHTS,
        "treatment": TREATMENT,
        "precedential": PRECEDENTIAL,
        "purpose": "internal precedent retrieval",
        "not_for": ["legal advice","court prediction","filing","settlement authority"]
    }, indent=2),
    encoding="utf-8"
)


In [ ]:
def tokens(text):
    return set(
        str(text).lower().replace("-"," ").replace("/"," ")
        .replace(","," ").replace("."," ").split()
    )

def issue_score(precedent, matter):
    p = tokens(precedent["legal_issue"] + " " + precedent["material_facts"])
    m = tokens(" ".join(
        matter["open_questions"] +
        matter["disputed_facts"] +
        matter["principal_defenses"] +
        [matter["cause_of_action"]]
    ))
    return round(100*len(p&m)/max(1,len(p|m)),2)

def recency(date_string):
    y = int(date_string[:4])
    return round(max(0,min(100,(y-2000)/26*100)),2)

def score(precedent, matter):
    comps = {
        "doctrinal_relevance": 100.0 if precedent["cause_of_action"] == matter["cause_of_action"] else 20.0,
        "issue_relevance": issue_score(precedent,matter),
        "jurisdictional_authority": PRECEDENTIAL[precedent["precedential_status"]]*0.70,
        "court_authority": precedent["court_level"]/3*100,
        "treatment_quality": TREATMENT[precedent["treatment_status"]],
        "recency": recency(precedent["decision_date"])
    }
    weighted = {k: round(comps[k]*WEIGHTS[k],4) for k in WEIGHTS}
    return {
        "precedent_id": precedent["precedent_id"],
        "matter_id": matter["matter_id"],
        "retrieval_score": round(sum(weighted.values()),2),
        "components": comps,
        "weighted_components": weighted,
        "precedential_status": precedent["precedential_status"],
        "treatment_status": precedent["treatment_status"],
        "court": precedent["court"],
        "jurisdiction": precedent["jurisdiction"],
        "decision_date": precedent["decision_date"],
        "legal_issue": precedent["legal_issue"],
        "holding": precedent["holding"],
        "rule_of_law": precedent["rule_of_law"]
    }

retrieval = {}
for matter in matters:
    scored = [score(p,matter) for p in precedents]
    scored.sort(key=lambda x:(-x["retrieval_score"],-PRECEDENTIAL[x["precedential_status"]],x["precedent_id"]))
    retrieval[matter["matter_id"]] = scored

(VAULT/"data"/"baby_step_1_retrieval_results.json").write_text(
    json.dumps(retrieval,indent=2),encoding="utf-8"
)
print("Retrieval complete.")


In [ ]:
def authority_sets(scored):
    supporting = [x for x in scored if x["treatment_status"] in ["Positive","Followed"]][:5]
    contrary = [x for x in scored if x["treatment_status"] in ["Distinguished","Limited","Criticized","Overruled"]][:5]
    similar = sorted(scored,key=lambda x:-x["components"]["issue_relevance"])[:5]
    disruptive = sorted(
        [x for x in scored if x["treatment_status"] in ["Limited","Criticized","Overruled"]],
        key=lambda x:x["decision_date"], reverse=True
    )[:5]
    return {
        "supporting": supporting,
        "contrary": contrary,
        "most_similar": similar,
        "newest_disruptive": disruptive
    }

authorities = {mid:authority_sets(items) for mid,items in retrieval.items()}
(VAULT/"data"/"baby_step_1_authority_sets.json").write_text(
    json.dumps(authorities,indent=2),encoding="utf-8"
)


## Strategy alternatives

Each matter receives three strategies. One becomes the preferred internal baseline; one remains the formal fallback; one is preserved as a rejected alternative.

The model evaluates legal support, procedural fit, evidentiary readiness, risk alignment, and institutional objective.


In [ ]:
TEMPLATES = {
"Breach of Shareholders Agreement":[
("Seek preliminary injunctive relief","Pursue expedited declaratory relief","Negotiate structured buyout or consent solution"),
],
"Post-M&A Purchase Price Dispute":[
("Enforce expert-determination mechanism","Seek early contract-interpretation ruling","Negotiate accounting-based settlement range"),
],
"Technology Licensing Dispute":[
("Seek targeted injunction and expedited technical discovery","Pursue contractual damages and audit rights","Renegotiate license and compliance framework"),
],
"Joint Venture Deadlock":[
("Invoke contractual deadlock and buy-sell mechanism","Seek judicial dissolution or equitable relief","Negotiate governance reset and staged buyout"),
],
"Supply Agreement Dispute":[
("Pursue liability-limitation and causation defenses","Seek early partial summary judgment","Negotiate continuity-based commercial settlement"),
]
}
TEMPLATES = {k:list(v[0]) for k,v in TEMPLATES.items()}

def support_delta(authority):
    s = statistics.mean([x["retrieval_score"] for x in authority["supporting"]]) if authority["supporting"] else 0
    c = statistics.mean([x["retrieval_score"] for x in authority["contrary"]]) if authority["contrary"] else 0
    return s-0.45*c

def evidence_readiness(m):
    return max(0,min(100,55+7*len(m["evidence_summary"])-5*len(m["open_questions"])))

def procedural_scores(posture):
    return {
        "Pre-Action Assessment":[88,74,82],
        "Pleadings":[78,87,72],
        "Document Production":[84,79,76],
        "Fact Discovery":[82,74,86],
        "Expert Discovery":[85,88,79]
    }.get(posture,[75,75,75])

def risk_scores(tolerance):
    return {
        "Low":[62,75,88],
        "Moderate":[84,80,82],
        "High":[92,84,72]
    }[tolerance]

evaluations = {}
for m in matters:
    mid = m["matter_id"]
    options = []
    for idx,name in enumerate(TEMPLATES[m["cause_of_action"]]):
        legal = max(0,min(100,60+support_delta(authorities[mid])))
        evidence = evidence_readiness(m)
        procedure = procedural_scores(m["procedural_posture"])[idx]
        risk = risk_scores(m["risk_tolerance"])[idx]
        objective = [90,82,86][idx]
        total = round(legal*.30+evidence*.20+procedure*.20+risk*.15+objective*.15,2)
        options.append({
            "strategy":name,
            "legal_support":round(legal,2),
            "evidence_readiness":round(evidence,2),
            "procedural_fit":procedure,
            "risk_alignment":risk,
            "objective_alignment":objective,
            "strategy_score":total
        })
    options.sort(key=lambda x:-x["strategy_score"])
    evaluations[mid] = options

(VAULT/"data"/"baby_step_1_strategy_evaluations.json").write_text(
    json.dumps(evaluations,indent=2),encoding="utf-8"
)
for mid,opts in evaluations.items():
    print(mid,opts[0]["strategy"],opts[0]["strategy_score"])


## Adversarial challenge

The preferred strategy is not accepted until the system records:

- contrary authority;
- unresolved factual assumptions;
- procedural uncertainty;
- evidentiary weakness;
- the fallback strategy;
- conditions that would require revision.


In [ ]:
challenges = {}
for m in matters:
    mid = m["matter_id"]
    contrary_ids = [x["precedent_id"] for x in authorities[mid]["contrary"][:3]]
    items = []
    if contrary_ids:
        items.append("Contrary authorities require analysis: " + ", ".join(contrary_ids))
    items.append("The evidentiary record remains synthetic and incomplete.")
    items.append("The preferred remedy depends on procedural posture and proof.")
    if m["risk_tolerance"] == "Low":
        items.append("Low risk tolerance favors a staged or narrowing approach.")
    items.append("The fallback strategy may become preferable if core assumptions fail.")
    challenges[mid] = {
        "matter_id":mid,
        "preferred_strategy":evaluations[mid][0]["strategy"],
        "fallback_strategy":evaluations[mid][1]["strategy"],
        "challenge_items":items,
        "conditional":len(items)>=4
    }

(VAULT/"data"/"baby_step_1_adversarial_challenges.json").write_text(
    json.dumps(challenges,indent=2),encoding="utf-8"
)


In [ ]:
recommendations = []
for m in matters:
    mid = m["matter_id"]
    preferred = evaluations[mid][0]
    fallback = evaluations[mid][1]
    challenge = challenges[mid]
    confidence = max(0,min(100,round(preferred["strategy_score"]-3*len(challenge["challenge_items"]),2)))
    recommendations.append({
        "recommendation_id":f"REC-{mid}-V001",
        "matter_id":mid,
        "version":1,
        "date":datetime.date.today().isoformat(),
        "preferred_strategy":preferred["strategy"],
        "fallback_strategy":fallback["strategy"],
        "rejected_alternative":evaluations[mid][2]["strategy"],
        "strategy_score":preferred["strategy_score"],
        "confidence_score":confidence,
        "conditional":challenge["conditional"],
        "supporting_authorities":[x["precedent_id"] for x in authorities[mid]["supporting"]],
        "contrary_authorities":[x["precedent_id"] for x in authorities[mid]["contrary"]],
        "most_similar_authorities":[x["precedent_id"] for x in authorities[mid]["most_similar"]],
        "factual_assumptions":m["disputed_facts"],
        "open_questions":m["open_questions"],
        "adversarial_challenges":challenge["challenge_items"],
        "permitted_next_actions":[
            "internal legal research",
            "internal evidence review",
            "strategy refinement",
            "committee-product preparation"
        ],
        "prohibited_actions":[
            "filing","service","party contact","court contact",
            "external counsel instruction","settlement offer",
            "external legal advice","external distribution"
        ],
        "human_decision_status":"Pending DEC-001",
        "synthetic":True
    })

assert len(recommendations)==5
(VAULT/"data"/"baby_step_1_recommendations_v1.json").write_text(
    json.dumps(recommendations,indent=2),encoding="utf-8"
)


In [ ]:
def write_note(path,lines):
    path.write_text("\n".join(lines).strip()+"\n",encoding="utf-8")

for r in recommendations:
    lines = [
        "---",
        f"recommendation_id: {r['recommendation_id']}",
        f"matter_id: {r['matter_id']}",
        "version: 1",
        "synthetic: true",
        f"conditional: {str(r['conditional']).lower()}",
        "---","",
        f"# {r['recommendation_id']}","",
        "## Matter","",
        f"[[../02_Active_Matters/{r['matter_id']}]]","",
        "## Preferred strategy","",r["preferred_strategy"],"",
        "## Fallback strategy","",r["fallback_strategy"],"",
        "## Rejected alternative","",r["rejected_alternative"],"",
        "## Confidence","",f"{r['confidence_score']}/100","",
        "## Supporting authorities",""
    ]
    lines += [f"- [[../01_Precedents/{pid}]]" for pid in r["supporting_authorities"]]
    lines += ["","## Contrary authorities",""]
    lines += [f"- [[../01_Precedents/{pid}]]" for pid in r["contrary_authorities"]] or ["- None"]
    lines += ["","## Most similar authorities",""]
    lines += [f"- [[../01_Precedents/{pid}]]" for pid in r["most_similar_authorities"]]
    lines += ["","## Factual assumptions",""]
    lines += [f"- {x}" for x in r["factual_assumptions"]]
    lines += ["","## Open questions",""]
    lines += [f"- {x}" for x in r["open_questions"]]
    lines += ["","## Adversarial challenges",""]
    lines += [f"- {x}" for x in r["adversarial_challenges"]]
    lines += ["","## Permitted next actions",""]
    lines += [f"- {x}" for x in r["permitted_next_actions"]]
    lines += ["","## Prohibited actions",""]
    lines += [f"- {x}" for x in r["prohibited_actions"]]
    lines += ["","## Temporal integrity","",
              "This is Recommendation Version 1 and must not be overwritten by later versions."]
    write_note(VAULT/"08_Recommendations"/f"{r['recommendation_id']}.md",lines)

print("Recommendation notes:",len(list((VAULT/"08_Recommendations").glob("REC-*-V001.md"))))


In [ ]:
brief = [
    "# Baby Step 1 — First Five-Matter Operating Brief","",
    "## Executive conclusion","",
    "The first strategy-development loop has produced five Recommendation V1 records.","",
    "## Recommendations",""
]
for r in recommendations:
    matter = next(m for m in matters if m["matter_id"]==r["matter_id"])
    brief += [
        f"### {matter['matter_id']} — {matter['caption']}","",
        f"- Preferred strategy: **{r['preferred_strategy']}**",
        f"- Fallback: {r['fallback_strategy']}",
        f"- Confidence: {r['confidence_score']}/100",
        f"- Conditional: {r['conditional']}",
        f"- Recommendation: [[../08_Recommendations/{r['recommendation_id']}]]",""
    ]
brief += [
    "## Requested decision","",
    "Accept Recommendation Version 1 as the internal analytical baseline.","",
    "## Not requested","",
    "- Filing authority","- Party or court contact","- Settlement authority",
    "- External legal advice","- External distribution"
]
write_note(VAULT/"10_Reports"/"Baby_Step_1_Operating_Brief.md",brief)


In [ ]:
DECISION = {
    "decision_id":"DEC-001",
    "date":datetime.date.today().isoformat(),
    "title":"Accept Recommendation Version 1 Baseline",
    "decision":"Accept the five Recommendation V1 records as the first internal strategy-development baseline.",
    "accepted_recommendations":[r["recommendation_id"] for r in recommendations],
    "permitted_next_actions":[
        "deeper internal legal research",
        "evidence refinement",
        "committee-product preparation",
        "preserve and compare future recommendation versions"
    ],
    "not_authorized":[
        "filing","service","party contact","court contact",
        "external counsel instruction","settlement offer",
        "external legal advice","external distribution"
    ],
    "synthetic":True
}
(VAULT/"09_Decisions"/"DEC-001.json").write_text(json.dumps(DECISION,indent=2),encoding="utf-8")

lines = [
    "# DEC-001 — Accept Recommendation Version 1 Baseline","",
    f"**Date:** {DECISION['date']}","",
    "## Decision","",DECISION["decision"],"",
    "## Accepted recommendations",""
]
lines += [f"- [[../08_Recommendations/{rid}]]" for rid in DECISION["accepted_recommendations"]]
lines += ["","## Permitted next actions",""]
lines += [f"- {x}" for x in DECISION["permitted_next_actions"]]
lines += ["","## Not authorized",""]
lines += [f"- {x}" for x in DECISION["not_authorized"]]
write_note(VAULT/"09_Decisions"/"DEC-001.md",lines)


In [ ]:
for m in matters:
    r = next(x for x in recommendations if x["matter_id"]==m["matter_id"])
    m["recommendation_status"] = r["recommendation_id"]
(VAULT/"data"/"active_matters.json").write_text(json.dumps(matters,indent=2),encoding="utf-8")

hot = [
    "# Current State — Hot Cache","",
    "## Current universe","",
    "- 1,000 synthetic precedents","- 5 active synthetic matters","",
    "## Current recommendations",""
]
hot += [f"- {r['matter_id']}: [[../08_Recommendations/{r['recommendation_id']}]]" for r in recommendations]
hot += [
    "","## Current decision","",
    "- [[../09_Decisions/DEC-001]]","",
    "## Permitted","",
    "- Internal research","- Evidence refinement","- Strategy comparison",
    "- Committee-product preparation","- Version preservation","",
    "## Prohibited","",
    "- Filing","- Service","- Party or court contact",
    "- Settlement offer","- External legal advice","- External distribution","",
    "## Next permitted experiment","",
    "Test whether the operating loop generalizes across all five heterogeneous matters."
]
write_note(VAULT/"12_Hot_Cache"/"Current_State.md",hot)


In [ ]:
errors = []
v1 = list((VAULT/"08_Recommendations").glob("REC-*-V001.md"))
v2 = list((VAULT/"08_Recommendations").glob("REC-*-V002.md"))
if len(v1)!=5:
    errors.append(f"Expected 5 Recommendation V1 notes, found {len(v1)}")
if v2:
    errors.append("Recommendation V2 exists prematurely")
for r in recommendations:
    if not r["supporting_authorities"]:
        errors.append(f"{r['recommendation_id']}: no supporting authorities")
    if not r["fallback_strategy"]:
        errors.append(f"{r['recommendation_id']}: no fallback strategy")
    if not r["prohibited_actions"]:
        errors.append(f"{r['recommendation_id']}: no prohibited actions")

required = [
    VAULT/"09_Decisions"/"DEC-001.md",
    VAULT/"10_Reports"/"Baby_Step_1_Operating_Brief.md",
    VAULT/"12_Hot_Cache"/"Current_State.md",
    VAULT/"data"/"baby_step_1_recommendations_v1.json"
]
for p in required:
    if not p.exists():
        errors.append(f"Missing: {p}")

validation = {
    "validated_at":datetime.datetime.now().isoformat(),
    "recommendation_v1_count":len(v1),
    "recommendation_v2_count":len(v2),
    "decision":"DEC-001",
    "errors":errors,
    "passed":len(errors)==0
}
(VAULT/"11_Audit"/"Baby_Step_1_Validation.json").write_text(
    json.dumps(validation,indent=2),encoding="utf-8"
)
assert validation["passed"],errors
print(json.dumps(validation,indent=2))
print("BABY STEP 1 PASSED")


In [ ]:
state.update({
    "completed_steps":sorted(set(state.get("completed_steps",[])+[1])),
    "current_step":1,
    "next_step":2,
    "recommendation_count":5,
    "current_recommendation_version":1,
    "decision":"DEC-001",
    "next_problem":"Test generalization across the five heterogeneous matters.",
    "permission_state":{
        "observe":True,
        "organize":True,
        "browse":True,
        "internal_strategy_analysis":True,
        "committee_product_preparation":True,
        "external_action":False
    }
})
state_path.write_text(json.dumps(state,indent=2),encoding="utf-8")

audit = {
    "timestamp":datetime.datetime.now().isoformat(),
    "step":1,
    "action":"Executed first five-matter strategy-development loop",
    "outputs":{"recommendations_v1":5,"decision":"DEC-001"},
    "validation_passed":True
}
with (VAULT/"11_Audit"/"workflow_audit.jsonl").open("a",encoding="utf-8") as f:
    f.write(json.dumps(audit)+"\n")
print(json.dumps(state,indent=2))
